# Decision Tree Classification | Diabetes Risk Prediction
### Professional ML engineering workshop — instructor edition

**Business question:** Can we use routine measurements to identify records associated with diabetes (`Outcome = 1`)?

**Model scope:** This notebook is a reproducible educational classification prototype, **not a clinically validated diagnostic model**.

**Outcomes of the workshop:** frame a binary classification problem; validate data; avoid leakage; construct a train-only imputation pipeline; establish baseline performance; understand Gini impurity and split selection; optimize hyperparameters using stratified cross-validation; evaluate an untouched test set; interpret tree decisions; discuss deployment risks.

**Suggested live-teaching pacing:** 10 min framing; 15 min data audit; 15 min tree theory; 20 min baseline pipeline; 20 min tuning and evaluation; 15 min explanation and model risks.

## 1 | The classification problem

Our **features** are `Pregnancies`, `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, `BMI`, `DiabetesPedigreeFunction`, `Age`.

Our **label** is `Outcome`: `0` = no diabetes, `1` = diabetes. The label is known during supervised training and hidden at inference time.

A classification model estimates a class label and optionally a **class probability**. The probability is a model score, not automatically a calibrated clinical risk probability.

**Instructor explanation:** A regression target is continuous (e.g., cost); a classification target consists of categories (e.g., Outcome 0/1). Here a *false negative* means a record with `Outcome = 1` is predicted as `0`—potentially important when screening. A *false positive* is the reverse, creating an unnecessary follow-up burden.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn import __version__ as sklearn_version
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, precision_score,
    recall_score, f1_score, roc_auc_score, average_precision_score,
    RocCurveDisplay, PrecisionRecallDisplay
)
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_validate, GridSearchCV
)
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text

RANDOM_STATE = 42
pd.set_option("display.max_columns", 30)
print(f"Python: {sys.version.split()[0]} | pandas: {pd.__version__} | scikit-learn: {sklearn_version}")

## 2 | Load your existing CSV

Place `diabetes-diabetes.csv` or `diabetes_extracted.csv` in the same directory as this notebook, **or** edit `DATA_PATH` below to point to the file. Your earlier class file was at `/Users/padideh/Documents/CICCC/ML/Week-5/diabetes-diabetes.csv`.

We do **not** download an undocumented replacement dataset: the workshop should use the actual course CSV. Some copies contain an extra `index` column; this is a row identifier, not a clinical predictor.

In [ ]:
CANDIDATE_PATHS = [
    Path("diabetes-diabetes.csv"),
    Path("diabetes_extracted.csv"),
    Path("/Users/padideh/Documents/CICCC/ML/Week-5/diabetes-diabetes.csv"),
]
DATA_PATH = next((path for path in CANDIDATE_PATHS if path.is_file()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Diabetes CSV not found. Put diabetes-diabetes.csv next to this notebook "
        "or add its path to CANDIDATE_PATHS."
    )
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df):,} rows from {DATA_PATH}")
print(df.head().to_string(index=False))

## 3 | Validate the data contract

An engineering-quality workflow verifies that it has the **right schema** and target values before training. Silent schema drift can lead to misleading output.

We intentionally exclude potential CSV-generated `Unnamed` and `index` fields. Such identifiers can give a model spurious correlations without adding meaningful medical information.

In [ ]:
FEATURES = [
    "Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
    "Insulin", "BMI", "DiabetesPedigreeFunction", "Age"
]
TARGET = "Outcome"
REQUIRED = FEATURES + [TARGET]
missing = sorted(set(REQUIRED).difference(df.columns))
if missing:
    raise ValueError(f"Missing required columns: {missing}")
if df[TARGET].isna().any():
    raise ValueError("Target contains missing values; investigate before training.")
for feature in FEATURES:
    df[feature] = pd.to_numeric(df[feature], errors="coerce")
labels = set(pd.to_numeric(df[TARGET], errors="coerce").dropna().unique())
if labels != {0, 1}:
    raise ValueError(f"Expected binary classes {{0,1}}; observed {labels}")
df[TARGET] = df[TARGET].astype(int)
print("Dataset shape:", df.shape)
print("Missing feature values:\n", df[FEATURES].isna().sum().to_string())
print("Class distribution:\n", df[TARGET].value_counts().sort_index().to_string())
print("Class proportions:\n", df[TARGET].value_counts(normalize=True).sort_index().round(3).to_string())
print("Duplicate complete records:", int(df[REQUIRED].duplicated().sum()))
print("Summary statistics:\n", df[FEATURES].describe().T.round(2).to_string())

## 4 | Domain-aware handling of zero measurements

Some physical measurements cannot realistically be zero in this context: `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, and `BMI`. In common versions of this dataset, zero is used as a **missing-value sentinel**.

We convert these specific zeros into `NaN`. This is a fixed rule based on feature meaning—not a statistic computed from the full dataset, so it does not leak test-set information. We **do not** replace zero in `Pregnancies`: zero pregnancies is valid. The training-only `SimpleImputer` then learns medians **inside** the pipeline.

**Critical nuance:** A zero may reflect a data collection or encoding problem; verify this interpretation against the actual data dictionary before clinical use.

In [ ]:
ZERO_AS_MISSING = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
working = df[REQUIRED].copy()
zero_counts = (working[ZERO_AS_MISSING] == 0).sum().sort_values(ascending=False)
print("Zero-coded counts before replacement:\n", zero_counts.to_string())
working[ZERO_AS_MISSING] = working[ZERO_AS_MISSING].replace(0, np.nan)
print("Missing after replacement:\n", working[FEATURES].isna().sum().to_string())

class_summary = working.groupby(TARGET)[FEATURES].median().T
print("Median feature values by target class:\n", class_summary.round(2).to_string())

## 5 | Split BEFORE fitting statistical preprocessing

A **holdout test set** simulates unseen future examples. We reserve 20% for final evaluation and fit all statistical transformations on the remaining 80% only.

`stratify=y` approximately preserves the positive/negative class proportions in both partitions, especially important for imbalanced classification.

Never use the test set to choose `max_depth`, thresholds, feature subsets, or other tuning decisions. Once test results are used to change the model, the test set is no longer genuinely untouched.

In [ ]:
X = working[FEATURES].copy()
y = working[TARGET].copy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
print(f"Train: {X_train.shape}, test: {X_test.shape}")
print(f"Train positive rate: {y_train.mean():.3f}; test positive rate: {y_test.mean():.3f}")

## 6 | How a Decision Tree actually learns

A tree recursively asks questions of the form `feature <= threshold?`, choosing a split that creates **purer** child nodes.

**Gini impurity** at a node with class probabilities \(p_0\) and \(p_1\):

\[G = 1 - (p_0^2 + p_1^2) = 2p_0p_1\]

- If every row belongs to one class, \(G=0\) (pure).
- If there is a 50/50 class mixture, \(G=0.5\) (maximum for binary classes).
- The algorithm evaluates candidates using the weighted child impurity: \(G_{split}=rac{n_L}{n}G_L + rac{n_R}{n}G_R\). It chooses the greatest impurity reduction.

For example, a node with 60% class 0 and 40% class 1 has Gini impurity \(1-(0.6^2+0.4^2)=0.48\). If a proposed split yields a much lower weighted child impurity, it is useful.

**Instructor distinction:** `gini` and `entropy` are split criteria; `max_depth`, `min_samples_leaf`, and pruning control complexity. Deeper trees can learn very fine patterns in training data yet generalize poorly (overfitting). Decision trees do **not** ordinarily need standardization because their splits depend on feature ordering, not scale or distances.

In [ ]:
def gini(proportion_positive: float) -> float:
    p = proportion_positive
    if not 0 <= p <= 1:
        raise ValueError("Probability must lie between 0 and 1")
    return 1 - (p ** 2 + (1 - p) ** 2)

for p in [0.0, 0.2, 0.4, 0.5, 0.8, 1.0]:
    print(f"Positive proportion={p:.1f} -> Gini impurity={gini(p):.3f}")

## 7 | Baseline and leakage-safe model pipeline

A professional model needs a meaningful reference point. `DummyClassifier(strategy="most_frequent")` always predicts the majority class, showing how misleading raw accuracy can be when class proportions are unequal.

The `Pipeline` applies imputation and then the Decision Tree. Its median imputation is learned from the training partition only. During cross-validation each fold fits its **own** median, preventing leakage across validation folds.

Start with a constrained tree (`max_depth=3`, `min_samples_leaf=10`) to emphasize interpretability; these are initial design choices, not claimed optimal values.

In [ ]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)
print("Majority-class baseline accuracy:", round(accuracy_score(y_test, baseline_pred), 3))
print("Majority-class baseline recall (Outcome=1):", round(recall_score(y_test, baseline_pred, zero_division=0), 3))

pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("tree", DecisionTreeClassifier(
        criterion="gini", max_depth=3, min_samples_leaf=10,
        random_state=RANDOM_STATE
    )),
])
pipeline.fit(X_train, y_train)
print("Training accuracy:", round(pipeline.score(X_train, y_train), 3))
print("Test accuracy:", round(pipeline.score(X_test, y_test), 3))

## 8 | Evaluate outcomes using complementary metrics

**Confusion matrix** (actual classes in rows; predicted classes in columns):

| | Predict 0 | Predict 1 |
|---|---:|---:|
| Actual 0 | True negative (TN) | False positive (FP) |
| Actual 1 | False negative (FN) | True positive (TP) |

**Accuracy:** \((TP+TN)/N\). **Precision:** \(TP/(TP+FP)\) — among predicted positives, how many are correct? **Recall / Sensitivity:** \(TP/(TP+FN)\) — among actual positives, how many do we catch? **Specificity:** \(TN/(TN+FP)\). **F1:** harmonic mean of precision and recall. **ROC AUC:** rank separation across thresholds. **Average precision:** summary of the precision–recall curve, which is often informative when positives are relatively rare.

Metrics have different operating implications: increasing sensitivity may increase unnecessary follow-ups. Choose trade-offs based on an explicitly stated use case and validation protocol.

In [ ]:
def evaluate_classifier(estimator, X_eval, y_eval, name="Classifier"):
    y_pred = estimator.predict(X_eval)
    scores = estimator.predict_proba(X_eval)[:, 1]
    tn, fp, fn, tp = confusion_matrix(y_eval, y_pred, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp) if tn + fp else float("nan")
    metrics = {
        "accuracy": accuracy_score(y_eval, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_eval, y_pred),
        "precision": precision_score(y_eval, y_pred, zero_division=0),
        "recall_sensitivity": recall_score(y_eval, y_pred, zero_division=0),
        "specificity": specificity,
        "f1": f1_score(y_eval, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_eval, scores),
        "average_precision": average_precision_score(y_eval, scores),
    }
    print(f"\n{name} — metric values")
    for metric, value in metrics.items():
        print(f"  {metric:<22} {value:.3f}")
    print("\nClassification report:\n", classification_report(
        y_eval, y_pred, target_names=["No diabetes", "Diabetes"], zero_division=0
    ))
    print(f"Confusion counts: TN={tn}, FP={fp}, FN={fn}, TP={tp}")
    return metrics

initial_metrics = evaluate_classifier(pipeline, X_test, y_test, "Initial constrained tree")
ConfusionMatrixDisplay.from_estimator(
    pipeline, X_test, y_test, display_labels=["No diabetes", "Diabetes"],
    values_format="d", cmap="Blues"
)
plt.title("Initial Decision Tree | Confusion Matrix")
plt.tight_layout()
plt.show()

## 9 | Visualize the fitted Decision Tree

Each node in the diagram shows a decision rule (`feature <= threshold`), impurity, sample count, and class counts. Left branches satisfy the displayed inequality. Right branches fail it. A leaf predicts the class supported by its learned class distribution.

**Avoid misreading the plot:** thresholds and feature ranking are learned from the provided dataset; they are not universal clinical guidelines. We use a small depth for readability.

In [ ]:
trained_tree = pipeline.named_steps["tree"]
plt.figure(figsize=(20, 10))
plot_tree(
    trained_tree, feature_names=FEATURES,
    class_names=["No diabetes", "Diabetes"],
    filled=True, rounded=True, fontsize=9, proportion=True
)
plt.title("Interpretable Decision Tree | Initial Model")
plt.tight_layout()
plt.show()

print("Text version of fitted rules:")
print(export_text(trained_tree, feature_names=FEATURES, decimals=2))

## 10 | Cross-validation: estimate generalization on training data

A single split can overstate or understate how well a model generalizes. **Stratified 5-fold cross-validation** divides the training set into five roughly class-balanced folds, fits on four folds and validates on the fifth, rotating five times.

Important: cross-validation here operates on **`X_train` only**; it does not touch the held-out test set. Because `pipeline` contains the imputer, each fold learns medians only from that fold's training subset.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_results = cross_validate(
    pipeline, X_train, y_train, cv=cv,
    scoring={"accuracy": "accuracy", "recall": "recall", "f1": "f1", "roc_auc": "roc_auc"},
    n_jobs=-1, return_train_score=True
)
for key in ["test_accuracy", "test_recall", "test_f1", "test_roc_auc"]:
    values = cv_results[key]
    print(f"{key:<20} mean={values.mean():.3f} | std={values.std():.3f} | folds={np.round(values, 3)}")
print("Mean training accuracy:", round(cv_results["train_accuracy"].mean(), 3))
print("Mean validation accuracy:", round(cv_results["test_accuracy"].mean(), 3))

## 11 | Hyperparameter optimization and model complexity

**`max_depth`** caps path length; **`min_samples_leaf`** ensures each terminal node has support; **`min_samples_split`** limits tiny branching groups; **`ccp_alpha`** penalizes overly complex trees using cost-complexity pruning. We test two split criteria as well.

We optimize **`roc_auc`** as a threshold-independent discrimination measure, then report recall, precision, and other performance separately. This is a pragmatic demonstration choice; a real clinical project should justify its metric from patient and operational costs.

Grid search uses 5-fold stratified CV on the training data. The best model is refit on the entire training set, *without* evaluating test performance during selection.

In [ ]:
param_grid = {
    "tree__criterion": ["gini", "entropy"],
    "tree__max_depth": [2, 3, 4, 5, 6],
    "tree__min_samples_leaf": [5, 10, 20],
    "tree__ccp_alpha": [0.0, 0.001, 0.005],
}
search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    refit=True,
    return_train_score=True,
)
search.fit(X_train, y_train)
best_model = search.best_estimator_
print("Best CV ROC AUC:", round(search.best_score_, 3))
print("Best hyperparameters:", search.best_params_)
results = pd.DataFrame(search.cv_results_).sort_values("rank_test_score")
print(results[["mean_test_score", "std_test_score", "mean_train_score", "params"]].head(10).to_string(index=False))

## 12 | Final holdout evaluation — one-time assessment

Only after model selection do we compute generalization metrics on `X_test`. Compare this score with cross-validation results. A large drop may indicate instability, selection bias, distribution differences, or limited sample size.

The ROC and precision–recall curves sweep possible classification thresholds; neither implies a clinically authorized threshold. On a small dataset, uncertainty can be substantial.

In [ ]:
final_metrics = evaluate_classifier(best_model, X_test, y_test, "Tuned Decision Tree")
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
ConfusionMatrixDisplay.from_estimator(
    best_model, X_test, y_test, display_labels=["No diabetes", "Diabetes"],
    cmap="Blues", values_format="d", ax=axes[0]
)
axes[0].set_title("Held-out confusion matrix")
RocCurveDisplay.from_estimator(best_model, X_test, y_test, ax=axes[1])
axes[1].set_title("ROC curve")
PrecisionRecallDisplay.from_estimator(best_model, X_test, y_test, ax=axes[2])
axes[2].set_title("Precision–recall curve")
plt.tight_layout()
plt.show()

## 13 | Explainability: global feature importance and local rules

A tree's **impurity-based feature importance** sums the weighted impurity reductions attributable to each variable. A larger value means the fitted tree relied more heavily on that variable for its splits. It **does not** prove causality or clinical importance. Impurity-based importance can favor continuous or high-cardinality features and is affected by correlated variables.

For a specific row, we can show the path through the tree. Predictions depend on the training data and the pipeline's imputed values, not on a manually chosen rule.

In [ ]:
importance = pd.Series(
    best_model.named_steps["tree"].feature_importances_, index=FEATURES
).sort_values(ascending=False)
print("Impurity-based feature importance:\n", importance.round(4).to_string())
plt.figure(figsize=(10, 4.5))
importance.sort_values().plot(kind="barh")
plt.xlabel("Mean decrease in impurity (relative)")
plt.title("Global Feature Importance — Tuned Decision Tree")
plt.tight_layout()
plt.show()

example = X_test.iloc[[0]].copy()
observed = y_test.iloc[0]
predicted = best_model.predict(example)[0]
score = best_model.predict_proba(example)[0, 1]
print("Example predictors:\n", example.to_string(index=False))
print(f"Actual label={observed}, predicted label={predicted}, estimated class-1 probability={score:.3f}")

imputed_example = best_model.named_steps["imputer"].transform(example)
tree = best_model.named_steps["tree"]
node_path = tree.decision_path(imputed_example).indices
for node_id in node_path:
    feat_idx = tree.tree_.feature[node_id]
    if feat_idx < 0:
        print(f"Leaf node {node_id}: prediction={predicted}")
        continue
    threshold = tree.tree_.threshold[node_id]
    val = imputed_example[0, feat_idx]
    symbol = "<=" if val <= threshold else ">"
    print(f"Node {node_id}: {FEATURES[feat_idx]}={val:.2f} {symbol} {threshold:.2f}")

## 14 | Reflection: what a professional reviewer should ask

1. **Data integrity:** Are zeros truly missing-value encodings? Is there a clinical data dictionary? Any duplicate subjects or records?
2. **Sampling:** Does the original dataset represent the population where the model will be used? Is there selection bias?
3. **Validation:** Are observations independent? If multiple rows come from the same individual or hospital, use group-aware validation. For forecasting a future period, use temporal validation.
4. **Imputation:** Does median imputation bias predictions? Should missingness indicators be added? Are missingness rates different across sites?
5. **Metric choice:** What matters more: missing positives or unnecessary alerts? Which costs and resource constraints determine the threshold?
6. **Calibration:** Are predicted class probabilities calibrated? An individual Decision Tree can be poorly calibrated; consider held-out calibration before treating scores as risks.
7. **Robustness and fairness:** Evaluate on external data and relevant population subgroups with appropriate governance.
8. **Model lifecycle:** Version the dataset, code, library dependencies, performance report, and model artifact. Monitor drift and changes in false-negative rates. Obtain clinical and regulatory review before deployment.

### Interview-quality summary

> We built a supervised binary classifier with an explicit data contract, domain-aware zero-to-missing conversion, train-only imputation, a stratified holdout, a majority-class baseline, cross-validation, hyperparameter tuning with cost-complexity pruning, and one-time test evaluation. We assessed accuracy, precision, sensitivity, specificity, F1, ROC AUC, and average precision. We interpreted the model's rules while acknowledging dataset representativeness, probability calibration, and deployment limitations.